# Participación Marzo 19

In [4]:
# Imports

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

## Cargar y Particionar Dataset

In [5]:
# Cargar conjunto de datos y particionar (80/10/10)

x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

In [9]:
# Obtener índices de columnas que se transformarán (se usó apoyo de la IA para esta parte)

cols_to_transform = [col for col in x.columns if col not in ['Latitude', 'Longitude']]
cols_passthrough = ['Latitude', 'Longitude']

cols = list(x.columns)

cols_to_transform_idx = [cols.index(c) for c in cols_to_transform]
cols_passthrough_idx = [cols.index(c) for c in cols_passthrough]

cols_to_transform_idx = torch.tensor(cols_to_transform_idx, dtype=torch.long)
cols_passthrough_idx = torch.tensor(cols_passthrough_idx, dtype=torch.long)

print(cols_to_transform_idx)
print(cols_passthrough_idx)

tensor([0, 1, 2, 3, 4, 5])
tensor([6, 7])


## Capa de Eliminación de Outliers

In [ ]:
# Definir variables para la transformación

Q1 = x_train[cols_to_transform].quantile(0.25)
Q3 = x_train[cols_to_transform].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

lower_bound = torch.tensor(lower_bound.values, dtype=torch.float32)
upper_bound = torch.tensor(upper_bound.values, dtype=torch.float32)


# Capa de eliminación de outliers

class RemoveOutliersLayer(nn.Module):
    def __init__(self, lower_bound: torch.Tensor, upper_bound: torch.Tensor, cols_to_transform_idx: torch.Tensor, cols_passthrough_idx: torch.Tensor):
        super().__init__()
        self.register_buffer('lower_bound', lower_bound)
        self.register_buffer('upper_bound', upper_bound)
        self.register_buffer('cols_to_transform_idx', cols_to_transform_idx)
        self.register_buffer('cols_passthrough_idx', cols_passthrough_idx)
    
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        x_transform = X[:, cols_to_transform_idx]
        x_pass = X[:, cols_passthrough_idx]

        x_transform = torch.clip(x_transform, self.lower_bound, self.upper_bound)

        x_out = torch.cat([x_transform, x_pass], dim=1)

        return x_out


## Capa de Escalamiento (Standard Scaling)